In [1]:
from typing import Tuple, List

import os
import rootutils

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

In [2]:
from src.utils import create_df, compute_fingerprints, compute_descriptors, create_data, eval_metrics, plot_pred_true, plot_importance

from src.avail_descriptors import descriptors_all, descriptors_short

In [3]:
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from argparse import Namespace

from sklearn.model_selection import train_test_split

---
# Feature Extraction:

In [5]:
data_path = "data/Bradley_dataset_ok_3"
columns = ['line_number', 'smiles', 'cas', 'label', 'T']

df = create_df(data_path, columns)

In [6]:
data_args = {
    "descriptors": descriptors_all,

    "apply_norm": True,
    
    "create_fingerprints": False,
    "temp_column": True,
}

### Computing Features (load, if already precomputed)

In [16]:
directory = 'saved_np_obj/Bradley'

if not os.path.exists(directory):
    os.makedirs(directory)

x_path = os.path.join(directory, 'X.npy')
labels_path = os.path.join(directory, 'labels.npy')
temp_path = os.path.join(directory, 'temp.npy')

if all(os.path.exists(path) for path in [x_path, labels_path, temp_path]):
    X = np.load(x_path)
    labels = np.load(labels_path)
    temp = np.load(temp_path)
else:
    _, X, labels, temp = create_data(df, **data_args)
    np.save(x_path, X)
    np.save(labels_path, labels)
    np.save(temp_path, temp)

---
# MultiTask Learning with Lightning:

In [1]:
from src.multitask_nn import MultiTaskModel, create_datasets

from lightning.pytorch.callbacks import Callback, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from lightning import seed_everything, Trainer
from torch.utils.data import DataLoader

from datetime import datetime

In [ ]:
seed_everything(42, verbose=False)

In [132]:
cfg = Namespace(
    project_name="bradley",
    
    train_size=0.8,

    batch_size=256,
    lr=1e-3,
    max_epochs=10,

    hid_dim=1024,
    drop=0.2,
    cl_loss_coef=1.,
    reg_loss_coef=1e-4,
)

In [133]:
train_data, val_data = create_datasets(X, labels, temp, train_size=cfg.train_size)

train_loader = DataLoader(train_data, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=cfg.batch_size, shuffle=False)

In [134]:
input_dim = X.shape[1]
model = MultiTaskModel(
    input_dim=input_dim,

    hidden_dim=cfg.hid_dim,
    drop=cfg.drop,

    lr=cfg.lr,

    cl_loss_coef=cfg.cl_loss_coef,
    reg_loss_coef=cfg.reg_loss_coef
)

current_date = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

logger = TensorBoardLogger(
    save_dir="tb_logs/", name=cfg.project_name
)

checkpoint_callback = ModelCheckpoint(
    dirpath=f"checkpoints/{current_date}-{cfg.project_name}",
    filename="{epoch:02d}-{val_loss:.4f}",
    save_top_k=1,
    monitor="V_tot",
    mode="min",
    save_last=True,
)

trainer = Trainer(
    logger=logger,
    callbacks=[checkpoint_callback],
    max_epochs=cfg.max_epochs,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [135]:
trainer.fit(model, train_loader, val_loader)


  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | shared       | Sequential        | 2.2 M  | train
1 | classifier   | Sequential        | 1.1 M  | train
2 | regressor    | Sequential        | 1.1 M  | train
3 | val_accuracy | BinaryAccuracy    | 0      | train
4 | val_f1       | BinaryF1Score     | 0      | train
5 | val_roc_auc  | BinaryAUROC       | 0      | train
6 | val_r2_class | R2Score           | 0      | train
7 | val_mse      | MeanSquaredError  | 0      | train
8 | val_mae      | MeanAbsoluteError | 0      | train
9 | val_r2_reg   | R2Score           | 0      | train
-----------------------------------------------------------
4.3 M     Trainable params
0         Non-trainable params
4.3 M     Total params
17.207    Total estimated model params size (MB)
27        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/avarlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/avarlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


---
# Plotting predictions: